# Notebook 07 — reviewed pilot readiness and source-locked intake

**Decision.** The [Notebook 06 protocol](../app/data/study_protocol_v1.json) still says `proposed_pending_faculty_and_ethics_review`. The [Notebook 06 report](../docs/validation/study_protocol.json) contains zero review rows and zero learner encounters. The repository contains no faculty approval of the protocol, synthetic cases, consent process, review roster, or outcome definitions and no institutional determination. **No prospective performance analysis runs here.** This notebook checks the import boundary and gives a concrete inventory of what is missing.

**Sources:** [`AGENTS.md`](../../AGENTS.md), [`context.md`](../../context.md) §§14–15, [`PRD.md`](../../PRD.md) §15 V-01–V-04, [Notebook 01 audit](../docs/validation/dataset_audit.json), [Notebook 02 report](../docs/validation/action_data.json), [Notebook 03 report](../docs/validation/patient_environment.json), [Notebook 04 report](../docs/validation/grounding_eval.json), [Notebook 05 report](../docs/validation/debrief_eval.json), and [Notebook 06 report](../docs/validation/study_protocol.json). All three authored case versions and shared `backend/` feedback/encounter contracts are inside the Notebook 06 source lock. HHS [coded-information guidance](https://www.hhs.gov/ohrp/coded-private-information-or-biospecimens-used-research.html) is a reference for the institutional determination; this code cannot decide oversight status.

In [ ]:
import os, sys, json
from pathlib import Path
os.environ["LLM_ENABLED"]="false"
ROOT=next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"PRD.md").exists())
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/"backend"))
from backend_kelsch.scripts.eval_integration import evaluate
from backend_kelsch.app.integration.quality import preflight, SCOPES, verify_audit_log
from backend_kelsch.app.study.protocol import lock_manifest
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
report=evaluate()
print("Mode",report["mode"],"Approval",report["approval"]["status"])
print("Source lock",report["source_lock"]["lock_id"])
assert report["gates"]["source_locked_import_quality"]=="PASS"
assert all(v=="NOT EVALUABLE" for k,v in report["gates"].items() if k!="source_locked_import_quality")

## 1. Approval inventory

Six decisions must be documented before a learner study: the protocol, synthetic case set, consent process, reviewer roster, outcome definitions, and institutional determination. No private document was provided. An optional `--approval-bundle` can check file digests, declared roles and the protocol lock, but **even complete metadata remains NOT VERIFIED**. An independent trusted coordinator must establish document authenticity and actual authority. The preflight deliberately rejects `--corpus` rather than treating a self-declared approval string as consent.

In [ ]:
display(pd.DataFrame([{"scope":scope,"source_checked":False,"institutionally_verified":False} for scope in SCOPES]))
print("Metadata checks:",report["approval"])
print("Learners imported:",report["prospective_learners"])
print("Confidence intervals:",report["confidence_intervals"],"Class balance:",report["reviewed_class_balance"])
print("Source hashes (selected):")
for p,h in list(report["source_lock"]["source_sha256"].items())[:8]: print(p,h[:16])

## 2. Blinded packet and review coverage

The source lock is compared with the Notebook 06 report before rebuilding the 287 pending reviewer packets. It also checks that release packets do not expose actual fact IDs or script level, semantic packets do not expose verifier verdicts, and rubric packets do not expose provisional scores. No script profile is an observed learner. Zero imported review rows means **missing 189/189 releases, 15/15 semantic labels, 41/41 claims and 42/42 domain ratings**. Disagreements and N/A ratings are zero *observed*, not proof of perfect agreement or no N/A domains.

In [ ]:
coverage=report["review_denominators"]
rows=[{"kind":k,"pending":coverage["missing_by_kind"][k],
       "resolved":coverage["resolved_by_kind"].get(k,0)} for k in coverage["total_by_kind"]]
display(pd.DataFrame(rows))
fig,ax=plt.subplots(figsize=(8,3.5))
ax.bar([r["kind"] for r in rows],[r["pending"] for r in rows],color="#bd6b50")
ax.set(ylabel="Missing independent resolution",title="Notebook 07 readiness: reviews still needed")
for i,r in enumerate(rows): ax.text(i,r["pending"]+2,str(r["pending"]),ha="center")
fig.tight_layout();plt.show()
print("Unresolved disagreements",coverage["unresolved_disagreements"],"Reviewed N/A",coverage["reviewed_na_domain_count"])
assert sum(r["resolved"] for r in rows)==0

## 3. Audit-chain quality control

When a private review JSONL is supplied, the shared Notebook 06 validator checks faculty role, distinct independent reviewers, exact spans, source hashes, adjudication and answer types. Notebook 07 recomputes the event chain and compares an optional audit export byte-for-byte with events derived from those review rows. The public report stores only counts and a chain head. A chain is tamper evident **relative to the provided review source**, not proof the reviewer is genuine or the institution approved the study. With no review source, the empty chain has a zero head and no events.

In [ ]:
print("Audit:",report["audit"])
print("Empty chain verified:",verify_audit_log([],expected_head="0"*64,packet_ids=set()))
print("Packet manifest digest:",report["packet_manifest_sha256"])
assert report["audit"]["events"]==0

## 4. Analysis readiness by claim

Notebook 06 already defines the statistical methods: whole-patient split and held-out baseline/prompted comparison, fact-release precision/recall, semantic errors, debrief support, quadratic weighted Cohen kappa with encounter-cluster bootstrap and a learner-level arm comparison. Notebook 07 intentionally **does not call those methods** without approved prospective inputs. The report uses `null` for class balance, intervals and performance failure examples, rather than zeros or fixture-derived labels. The three authored patient cases are too few independent patient groups for credible supervised extraction, and six authored debrief encounters cannot meet the proposed 20-per-domain agreement floor.

In [ ]:
display(pd.DataFrame([{"gate":k,"status":v} for k,v in report["gates"].items()]))
print("Performance failure examples:",report["performance_failure_examples"])
print("Reported limitations:")
for x in report["engineering_limitations"]:print("-",x)

## Run and next decisions

From the repository root: `LLM_ENABLED=false PYTHONPATH=backend python -m backend_kelsch.scripts.eval_integration`. `--reviews private.jsonl --audit-log private_audit.jsonl` checks imported review quality but does not report performance. `--approval-bundle private_bundle.json` checks referenced document digests and still says NOT VERIFIED. `--corpus private_corpus.json` fails closed while trusted institutional authorization is unavailable. Keep private learner transcripts, review rows and consent linkage outside git. See [machine report](../docs/validation/integration_readiness.json), [readable report](../docs/validation/integration_readiness.md) and [Notebook 08 prompt](../docs/notebook_08_prompt.txt).

**PASS:** the source-locked, blinded import and audit-quality invariants hold. **FAIL:** any lock mismatch, leaked answer, invalid approval digest or broken audit chain. **NOT EVALUABLE:** every clinical, model, grader, difficulty and learning gate until independent review, approved consent and a suitable learner sample are verified. The seven HTTP tests are separately reported with actual local status; a collected or timed-out test is not a pass.